# Case: Empresas Unicórnio — Análise Exploratória com Python

Este notebook analisa uma base histórica de empresas unicórnio com foco em:

- concentração geográfica;
- setores mais representativos;
- evolução anual de novos unicórnios;
- maiores valuations;
- investidores mais recorrentes;
- recorte das empresas brasileiras.

> **Importante:** o dataset é histórico e contém registros até 7 de dezembro de 2021.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

df = pd.read_csv("../data/raw/startups_2021_end.csv")
df.head()

## 1. Diagnóstico inicial

In [ ]:
print("Dimensões:", df.shape)
print("\nTipos:")
print(df.dtypes)
print("\nValores ausentes:")
print(df.isna().sum())
print("\nDuplicatas completas:", df.duplicated().sum())
print("Nomes de empresas repetidos:", df["Company"].duplicated().sum())

### Observação sobre duplicidade

Há um nome de empresa repetido: **Bolt**. Ele não deve ser removido automaticamente, porque representa empresas distintas em países e setores diferentes. Esse é um exemplo de por que a limpeza de dados não deve depender apenas de regras mecânicas.


## 2. Limpeza e transformação

In [ ]:
df = df.drop(columns=["Unnamed: 0"], errors="ignore").copy()

df["Valuation_B"] = (
    df["Valuation ($B)"]
    .str.replace("$", "", regex=False)
    .str.replace("B", "", regex=False)
    .astype(float)
)

df["Date_Joined"] = pd.to_datetime(df["Date Joined"])
df["Join_Year"] = df["Date_Joined"].dt.year
df["City"] = df["City"].fillna("Não informado")
df["Select Investors"] = df["Select Investors"].fillna("Não informado")

df.head()

## 3. Visão geral

In [ ]:
summary = pd.Series({
    "Empresas": len(df),
    "Países": df["Country"].nunique(),
    "Cidades": df["City"].replace("Não informado", pd.NA).nunique(),
    "Setores": df["Industry"].nunique(),
    "Valuation total (US$ bi)": df["Valuation_B"].sum(),
    "Valuation mediano (US$ bi)": df["Valuation_B"].median(),
    "Novos unicórnios em 2021": (df["Join_Year"] == 2021).sum(),
})
summary

## 4. Quais países concentram mais unicórnios?

In [ ]:
top_countries = df["Country"].value_counts().head(10)
top_countries.sort_values().plot(kind="barh", figsize=(10, 6), title="Top 10 países por número de unicórnios")
plt.xlabel("Número de empresas")
plt.ylabel("País")
plt.tight_layout()
plt.show()

## 5. Quais setores aparecem com maior frequência?

In [ ]:
top_industries = df["Industry"].value_counts().head(10)
top_industries.sort_values().plot(kind="barh", figsize=(10, 6), title="Top 10 setores")
plt.xlabel("Número de empresas")
plt.ylabel("Setor")
plt.tight_layout()
plt.show()

## 6. Como evoluiu a entrada de novos unicórnios?

In [ ]:
by_year = df["Join_Year"].value_counts().sort_index()
by_year.plot(kind="line", marker="o", figsize=(10, 6), title="Novos unicórnios por ano")
plt.xlabel("Ano")
plt.ylabel("Número de empresas")
plt.tight_layout()
plt.show()

## 7. Quais empresas possuem os maiores valuations?

In [ ]:
top_valuations = df.nlargest(10, "Valuation_B")[
    ["Company", "Valuation_B", "Country", "Industry"]
]
top_valuations

In [ ]:
(
    top_valuations
    .set_index("Company")["Valuation_B"]
    .sort_values()
    .plot(kind="barh", figsize=(10, 6), title="Top 10 empresas por valuation")
)
plt.xlabel("Valuation (US$ bilhões)")
plt.ylabel("Empresa")
plt.tight_layout()
plt.show()

## 8. Quais investidores aparecem com mais frequência?

In [ ]:
investors = (
    df["Select Investors"]
    .replace("Não informado", pd.NA)
    .dropna()
    .str.split(",")
    .explode()
    .str.strip()
)

top_investors = investors.value_counts().head(15)
top_investors

## 9. Recorte: unicórnios brasileiros

In [ ]:
brazil = (
    df[df["Country"] == "Brazil"]
    .sort_values("Valuation_B", ascending=False)
)

brazil[["Company", "Valuation_B", "City", "Industry", "Date_Joined"]]

In [ ]:
(
    brazil
    .set_index("Company")["Valuation_B"]
    .sort_values()
    .plot(kind="barh", figsize=(10, 7), title="Unicórnios brasileiros no dataset")
)
plt.xlabel("Valuation (US$ bilhões)")
plt.ylabel("Empresa")
plt.tight_layout()
plt.show()

## 10. Principais achados

Com base neste snapshot histórico:

1. Os **Estados Unidos** concentram aproximadamente metade das empresas da base.
2. **Fintech** é o setor com maior número de unicórnios, seguido por *Internet software & services*.
3. **2021** apresenta uma forte aceleração, com 480 empresas atingindo status de unicórnio no ano.
4. O valuation é bastante concentrado: empresas como Bytedance, SpaceX e Stripe ficam muito acima da mediana.
5. O Brasil possui **15 empresas** no dataset; Nubank se destaca com valuation de US$ 30 bilhões no snapshot.
6. Investidores como Accel, Sequoia e Tiger Global aparecem repetidamente entre as empresas da base.

### Limitações

- O dataset é um retrato histórico, não uma lista atual de unicórnios.
- Valuation não representa receita, lucro ou valor realizado em uma saída.
- A coluna `Select Investors` lista investidores selecionados, não necessariamente todos os investidores.
- Algumas cidades e um registro de investidores estavam ausentes na fonte.
